# Full two-stage StackGAN generation

This notebook generates a 64×64 Stage-I sketch and its 256×256 Stage-II refinement from the completed `stage2_bbox` model.

> The repository provides precomputed char-CNN/RNN embeddings, not the text encoder. Enter an **exact** caption from `birds/text_c10`; arbitrary new text cannot be embedded by this model.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'models.py').is_file():
    raise RuntimeError('Start Jupyter from the project root containing models.py.')
sys.path.insert(0, str(PROJECT_ROOT))

import torch
from PIL import Image
from IPython.display import display

import numpy as np
import pandas as pd
from models import Stage1_Generator, Stage2_Generator

TEXT_ROOT = PROJECT_ROOT / 'birds' / 'text_c10'
DEFAULT_PROMPT = 'this bird has a bright yellow body, with brown on its crown and wings.'

CHECKPOINT_PATH = PROJECT_ROOT / 'runs' / 'stage2_bbox' / 'best_train.pt'
OUTPUT_DIR = PROJECT_ROOT / 'generated' / 'two_stage_notebook'
SEED = 42  # Change to produce another variation for the same caption.
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(f'Checkpoint not found: {CHECKPOINT_PATH}')
print(f'Using {DEVICE}; checkpoint: {CHECKPOINT_PATH}')


In [ ]:
def normalize_caption(text: str) -> str:
    return ' '.join(text.casefold().split())

def build_caption_index():
    """Map normalized CUB captions to (split, image index, caption index, original text)."""
    image_locations = {}
    for split in ('train', 'test'):
        filenames = pd.read_pickle(PROJECT_ROOT / 'birds' / split / 'filenames.pickle')
        for image_index, filename in enumerate(filenames):
            image_locations[filename.replace('\\', '/')] = (split, image_index)

    index = {}
    for text_path in TEXT_ROOT.rglob('*.txt'):
        filename = text_path.relative_to(TEXT_ROOT).with_suffix('').as_posix()
        if filename not in image_locations:
            continue
        split, image_index = image_locations[filename]
        captions = text_path.read_text(encoding='utf-8').splitlines()
        for caption_index, caption in enumerate(captions[:10]):
            index.setdefault(normalize_caption(caption), (split, image_index, caption_index, caption))
    return index


def find_embedding(prompt, device):
    index = build_caption_index()
    key = normalize_caption(prompt)
    if key not in index:
        raise ValueError('No exact caption embedding exists. Use a caption from birds/text_c10.')
    split, image_index, caption_index, caption = index[key]
    embeddings = np.asarray(
        pd.read_pickle(PROJECT_ROOT / 'birds' / split / 'char-CNN-RNN-embeddings.pickle'),
        dtype=np.float32,
    )
    embedding = torch.from_numpy(embeddings[image_index, caption_index]).unsqueeze(0).to(device)
    return embedding, caption


def load_generator(path, device):
    checkpoint = torch.load(path, map_location=device, weights_only=False)
    args = checkpoint.get('args', {})
    stage1 = Stage1_Generator(
        noise_size=args.get('noise_size', 100),
        base_channels=args.get('base_channels', 64),
    )
    generator = Stage2_Generator(
        1024, 128, stage1, base_channels=args.get('base_channels', 64),
    ).to(device)
    state = checkpoint['generator_ema'] if 'generator_ema' in checkpoint else checkpoint['generator']
    generator.load_state_dict(state)
    generator.eval()
    print(f"Loaded checkpoint epoch {checkpoint.get('epoch', 'unknown')}.")
    return generator


def to_pil(image):
    pixels = ((image.detach().cpu().clamp(-1, 1) + 1) * 127.5).round().to(torch.uint8)
    return Image.fromarray(pixels.permute(1, 2, 0).numpy())


In [ ]:
# Supply an exact caption from birds/text_c10.
PROMPT = input(f'Caption [{DEFAULT_PROMPT}]: ').strip() or DEFAULT_PROMPT
embedding, matched_caption = find_embedding(PROMPT, DEVICE)
print(f'Matched caption: {matched_caption}')


In [ ]:
generator = load_generator(CHECKPOINT_PATH, DEVICE)
torch.manual_seed(SEED)
if DEVICE.type == 'cuda':
    torch.cuda.manual_seed_all(SEED)

with torch.no_grad():
    stage2_image, stage1_image, _, _ = generator(embedding)

stage1_pil = to_pil(stage1_image[0])
stage2_pil = to_pil(stage2_image[0])

comparison = Image.new('RGB', (512, 256))
comparison.paste(stage1_pil.resize((256, 256), Image.Resampling.NEAREST), (0, 0))
comparison.paste(stage2_pil, (256, 0))
display(comparison)


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
stage1_pil.save(OUTPUT_DIR / 'stage1_64.png')
stage2_pil.save(OUTPUT_DIR / 'stage2_256.png')
comparison.save(OUTPUT_DIR / 'comparison.png')
print(f'Saved images in: {OUTPUT_DIR}')
